In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [30]:
drought_df = pd.read_csv('tableExport.csv')
drought_df['date'] = pd.to_datetime(drought_df['Week'])
daily_dates = pd.DataFrame({
    'date': pd.date_range(df_features['date'].min(), df_features['date'].max(), freq='D')
})
drought_df.head()

,Week,None,D0,D1,D2,D3,D4,DSCI,date
0,2026-04-21,0.0,0.0,0.00,100.00,0.0,0.0,300,2026-04-21
1,2026-04-14,0.0,0.0,0.00,100.00,0.0,0.0,300,2026-04-14
2,2026-04-07,0.0,0.0,0.00,100.00,0.0,0.0,300,2026-04-07
3,2026-03-31,0.0,0.0,65.86,34.14,0.0,0.0,234,2026-03-31
4,2026-03-24,0.0,0.0,65.97,34.03,0.0,0.0,234,2026-03-24


In [43]:
drought_daily = daily_dates.merge(drought_df, on='date', how='left')
drought_daily = drought_daily.sort_values('date')
drought_daily = drought_daily.ffill()
drought_daily['severe_drought_pct'] = (
    drought_daily['D2'] + drought_daily['D3'] + drought_daily['D4']
)
drought_daily = drought_daily[['date', 'DSCI', 'severe_drought_pct']]
drought_daily['date'] = pd.to_datetime(drought_daily['date'])

In [44]:
drought_daily.describe()

,date,DSCI,severe_drought_pct
count,6280,6279.000000,6279.000000
mean,2016-05-19 12:00:00,44.593088,2.124404
min,2007-10-15 00:00:00,0.000000,0.000000
25%,2012-01-31 18:00:00,0.000000,0.000000
50%,2016-05-19 12:00:00,0.000000,0.000000
75%,2020-09-05 06:00:00,100.000000,0.000000
max,2024-12-23 00:00:00,375.000000,100.000000
std,NaN,72.503638,13.043081


In [48]:
raw = pd.read_csv('merged_with_gage.csv', index_col='Unnamed: 0')
raw = raw[['date', 'discharge', 'precipitation', 'gage_height']]
raw['date'] = pd.to_datetime(raw['date'])
raw.head()

,date,discharge,precipitation,gage_height
0,2007-10-01,37.9,0.0,2.151711
1,2007-10-02,37.1,0.0,2.146250
2,2007-10-03,36.4,0.0,2.140521
3,2007-10-04,37.6,0.0,2.150000
4,2007-10-05,38.1,0.0,2.150937


In [49]:
raw = raw.merge(drought_daily, on='date')
raw = raw.dropna()

In [50]:
raw.head()

,date,discharge,precipitation,gage_height,DSCI,severe_drought_pct
1,2007-10-16,32.7,0.00,2.105417,300.0,100.0
2,2007-10-17,35.8,0.00,2.129896,300.0,100.0
3,2007-10-18,37.7,0.00,2.152083,300.0,100.0
4,2007-10-19,38.2,7.45,2.149271,300.0,100.0
5,2007-10-20,51.8,0.00,2.179583,300.0,100.0


In [52]:
def make_features(df, horizons=[1, 3, 7]):
    """
    Makes lagged, rolling window, rate of change, and calendar features.
    
    """

    df = df.copy().sort_values('date').reset_index(drop=True)

    # Lag features
    lag_configs = {
        'gage_height': [1, 2, 3, 7, 14],
        'discharge': [1, 2, 3, 7, 14],
        'precipitation': [1, 2, 3, 7]
    }

    for col, lags in lag_configs.items():
        for lag in lags:
            df[f'{col}_lag{lag}'] = df[col].shift(lag)

    # DSCI-specific features
    for lag in [7, 14, 30]:
        df[f'DSCI_lag{lag}'] = df['DSCI'].shift(lag)

    df['DSCI_roll30_mean'] = df['DSCI'].shift(1).rolling(30).mean()

    
    # Rolling window features
    roll_configs = {
        'gage_height': {'mean': [3, 7, 14], 'max': [3, 7]},
        'discharge': {'mean': [3, 7, 14], 'max': [3, 7]},
        'precipitation': {'mean': [3, 7, 14], 'max': [3, 7], 'sum': [3, 7, 14]}
    }

    for col, ops in roll_configs.items():
        for op, windows in ops.items():
            for window in windows:
                rolled = df[col].shift(1).rolling(window)
                df[f'{col}_roll{window}_{op}'] = getattr(rolled, op)()

    # Rate of change features
    for col in ['gage_height', 'discharge']:
        df[f'{col}_diff1'] = df[col].diff(1)
        df[f'{col}_diff3'] = df[col].diff(3)

    # Calendar/seasonal features
    season_map = {
        12: 0, 1: 0, 2: 0,   # Winter
         3: 1, 4: 1, 5: 1,   # Spring
         6: 2, 7: 2, 8: 2,   # Summer
         9: 3, 10: 3, 11: 3  # Fall
    }
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['season'] = df['month'].map(season_map)

    # Forecast targets
    targets = {}
    for h in horizons:
        col = f'gage_height_t{h}'
        df[col] = df['gage_height'].shift(-h)
        targets[h] = col

    # Clean up
    df = df.dropna().reset_index(drop=True)

    return df, targets

In [53]:
df_features, targets = make_features(raw)

In [56]:
df_features.shape

(6231, 52)

In [57]:
df_features.head()

,date,discharge,precipitation,gage_height,DSCI,severe_drought_pct,gage_height_lag1,gage_height_lag2,gage_height_lag3,gage_height_lag7,...,gage_height_diff1,gage_height_diff3,discharge_diff1,discharge_diff3,month,day_of_year,season,gage_height_t1,gage_height_t3,gage_height_t7
0,2007-11-15,110.0,6.48,2.525521,200.0,0.0,2.507396,2.508437,2.523229,2.467917,...,0.018125,0.002292,5.0,3.0,11,319,3,2.605729,2.515625,2.492500
1,2007-11-16,136.0,0.00,2.605729,200.0,0.0,2.525521,2.507396,2.508437,2.439792,...,0.080208,0.097292,26.0,32.0,11,320,3,2.573542,2.490000,2.508750
2,2007-11-17,118.0,0.00,2.573542,200.0,0.0,2.605729,2.525521,2.507396,2.451354,...,-0.032188,0.066146,-18.0,13.0,11,321,3,2.515625,2.490000,2.489792
3,2007-11-18,103.0,0.00,2.515625,200.0,0.0,2.573542,2.605729,2.525521,2.514063,...,-0.057917,-0.009896,-15.0,-7.0,11,322,3,2.490000,2.491458,2.474167
4,2007-11-19,99.5,0.00,2.490000,200.0,0.0,2.515625,2.573542,2.605729,2.523229,...,-0.025625,-0.115729,-3.5,-36.5,11,323,3,2.490000,2.492500,2.480104


In [58]:
df_features.to_csv('final_w_features.csv', index=False)

In [59]:
foo = pd.read_csv('final_w_features.csv')
foo.head()

,date,discharge,precipitation,gage_height,DSCI,severe_drought_pct,gage_height_lag1,gage_height_lag2,gage_height_lag3,gage_height_lag7,...,gage_height_diff1,gage_height_diff3,discharge_diff1,discharge_diff3,month,day_of_year,season,gage_height_t1,gage_height_t3,gage_height_t7
0,2007-11-15,110.0,6.48,2.525521,200.0,0.0,2.507396,2.508437,2.523229,2.467917,...,0.018125,0.002292,5.0,3.0,11,319,3,2.605729,2.515625,2.492500
1,2007-11-16,136.0,0.00,2.605729,200.0,0.0,2.525521,2.507396,2.508437,2.439792,...,0.080208,0.097292,26.0,32.0,11,320,3,2.573542,2.490000,2.508750
2,2007-11-17,118.0,0.00,2.573542,200.0,0.0,2.605729,2.525521,2.507396,2.451354,...,-0.032188,0.066146,-18.0,13.0,11,321,3,2.515625,2.490000,2.489792
3,2007-11-18,103.0,0.00,2.515625,200.0,0.0,2.573542,2.605729,2.525521,2.514063,...,-0.057917,-0.009896,-15.0,-7.0,11,322,3,2.490000,2.491458,2.474167
4,2007-11-19,99.5,0.00,2.490000,200.0,0.0,2.515625,2.573542,2.605729,2.523229,...,-0.025625,-0.115729,-3.5,-36.5,11,323,3,2.490000,2.492500,2.480104


In [25]:
targets

{1: 'gage_height_t1', 3: 'gage_height_t3', 7: 'gage_height_t7'}

In [60]:
print(f"Shape after feature engineering: {df_features.shape}")
print(f"\nTarget columns: {list(targets.values())}")
print(f"\nFeature columns ({len(df_features.columns) - len(targets) - 2}):")
feature_cols = [c for c in df_features.columns 
                if c not in list(targets.values()) + ['date', 'gage_height']]
print(feature_cols)

Shape after feature engineering: (6231, 52)

Target columns: ['gage_height_t1', 'gage_height_t3', 'gage_height_t7']

Feature columns (47):
['discharge', 'precipitation', 'DSCI', 'severe_drought_pct', 'gage_height_lag1', 'gage_height_lag2', 'gage_height_lag3', 'gage_height_lag7', 'gage_height_lag14', 'discharge_lag1', 'discharge_lag2', 'discharge_lag3', 'discharge_lag7', 'discharge_lag14', 'precipitation_lag1', 'precipitation_lag2', 'precipitation_lag3', 'precipitation_lag7', 'DSCI_lag7', 'DSCI_lag14', 'DSCI_lag30', 'DSCI_roll30_mean', 'gage_height_roll3_mean', 'gage_height_roll7_mean', 'gage_height_roll14_mean', 'gage_height_roll3_max', 'gage_height_roll7_max', 'discharge_roll3_mean', 'discharge_roll7_mean', 'discharge_roll14_mean', 'discharge_roll3_max', 'discharge_roll7_max', 'precipitation_roll3_mean', 'precipitation_roll7_mean', 'precipitation_roll14_mean', 'precipitation_roll3_max', 'precipitation_roll7_max', 'precipitation_roll3_sum', 'precipitation_roll7_sum', 'precipitation_rol